# OCR demo — grant-administration documents

A walkthrough of the full extraction pipeline for university grant docs
(award notices, budgets, proposals) using a Qwen-VL vision-language model
deployed on RunAI.

The pipeline has three pieces:

1. **Render** each PDF page as an image (PyMuPDF, ~144 DPI).
2. **Extract** structured JSON from each image via a deployed vLLM endpoint
   running Qwen3-VL-32B. Pages are sent in chunks of up to ~20 with a 1-page
   sliding overlap so the model can reason about cross-page tables and
   narratives.
3. **Merge + synthesize** chunk outputs into one document-level JSON.
   Pass 2 adds a short doc-level summary and lints for issues that the
   per-chunk extraction can't see (continuation flags still set, possible
   duplicate stakeholders, etc.).

The shared pipeline lives in `ocr_app/scripts/` — this notebook drives it
interactively so you can watch each stage. The same pieces run unchanged
inside the production batch script (`ocr_app/scripts/batch_extract.py`).

> **Library / archival materials?** Use `library_extraction_pipeline_demo.ipynb` —
> same architecture, library-specific prompt and schema.


## 1. Connect to the deployed vLLM endpoint

The model runs as a separate RunAI workload exposing an OpenAI-compatible
HTTP endpoint. This notebook is a thin client that talks to it.

The cell below picks the right URL based on **`RUN_LOCATION`** at the top
— `in_cluster` for a RunAI workspace, `laptop_vpn` for a laptop on the
campus VPN — auto-detects the served model, and runs a smoke test.


In [ ]:
import os
import httpx

# ── Pick where you're running ─────────────────────────────────────────
# "in_cluster" — RunAI workspace; calls the cluster-internal service hostname
# "laptop_vpn" — laptop on the campus VPN; calls the public Knative hostname
# Set the VLLM_BASE_URL env var to override either default.
#
# Note: "laptop_vpn" goes through the Knative ingress, which has a ~1 MB
# request-body cap. The notebook automatically downscales + re-encodes
# page images as JPEG on this path so the body fits; in-cluster runs send
# full-resolution PNGs since they bypass the ingress.
RUN_LOCATION = "laptop_vpn"

URLS = {
    "in_cluster": "http://qwen3--vl--32b--instruct-awq.runai-shared-models.svc.cluster.local/v1",
    "laptop_vpn": "https://qwen3--vl--32b--instruct-awq-runai-shared-models.deepthought.doit.wisc.edu/v1",
}

VLLM_BASE_URL = os.environ.get("VLLM_BASE_URL", URLS[RUN_LOCATION])
# Per-call output budget. The grant-admin demo bumps this to fit dense 10–20
# page chunks; the library demo runs page-by-page so a smaller budget is fine.
VLLM_MAX_TOKENS = 90000
MODEL_NAME = os.environ.get("VLM_MODEL", "QuantTrio/Qwen3-VL-32B-Instruct-AWQ")

# Auto-detect the served model + run a smoke test. Wrapped in try/except so
# a failure here (wrong RUN_LOCATION, VPN off, server down) prints cleanly
# but leaves VLLM_BASE_URL / MODEL_NAME / VLLM_MAX_TOKENS defined for the
# rest of the notebook.
try:
    resp = httpx.get(f"{VLLM_BASE_URL}/models", timeout=10)
    resp.raise_for_status()
    MODEL_NAME = resp.json()["data"][0]["id"]
    print(f"Mode:     {RUN_LOCATION}")
    print(f"Endpoint: {VLLM_BASE_URL}")
    print(f"Model:    {MODEL_NAME}")
    print(f"HF card:  https://huggingface.co/{MODEL_NAME}")

    resp = httpx.post(
        f"{VLLM_BASE_URL}/chat/completions",
        json={
            "model": MODEL_NAME,
            "messages": [{"role": "user", "content": "In a single sentence, share with me the entire life story of Winnie the Pooh."}],
            "max_tokens": 100,
            "temperature": 0.0,
        },
        timeout=60,
    )
    resp.raise_for_status()
    print("Smoke test:", resp.json()["choices"][0]["message"]["content"].strip())
except Exception as e:
    print(f"Endpoint unreachable: {type(e).__name__}: {e}")
    print(f"Defaults still set: VLLM_BASE_URL={VLLM_BASE_URL!r}, MODEL_NAME={MODEL_NAME!r}")
    print("Switch RUN_LOCATION above (or set VLLM_BASE_URL env var) and re-run.")


### Running locally?

If you're on the laptop / VPN path, set `RUN_LOCATION = "laptop_vpn"` in
the cell above. A fresh laptop env also won't have the imports this
notebook uses (`httpx`, `pymupdf` / `fitz`, `Pillow`). Quick setup with
[uv](https://docs.astral.sh/uv/) — last line registers the venv as a
JupyterLab kernel named **OCR app (uv venv)** so this notebook
auto-selects it on open:

```bash
uv venv && source .venv/bin/activate
uv pip install httpx pymupdf Pillow ipykernel
python -m ipykernel install --user --name=ocr-app --display-name="OCR app (uv venv)"
```

(or the equivalent `python -m venv` + `pip install` if you're not on uv.)

The endpoint is OpenAI-compatible — `httpx` is what `run_vlm` uses below,
but the `openai` Python client works against the same URL if you prefer.
See [`docs/deploy-vllm.md` → Access from outside the cluster (VPN)](../docs/deploy-vllm.md#access-from-outside-the-cluster-vpn)
for details on the public hostname pattern.


## 2. Pull in the shared pipeline pieces

The notebook reuses three things from `ocr_app/scripts/` so we don't
duplicate logic that the production batch script already has:

- `chunk_page_ranges` — splits an N-page document into overlapping ranges.
- `build_chunk_messages` — builds the multi-image VLM prompt for a chunk
  (handles sliding-window context, pinned cover page, forward-context hint).
- `merge_chunks` — stitches per-chunk extractions back into one doc-level
  JSON, deduping cross-chunk tables, narratives, stakeholders, etc.
- `DOC_SYNTHESIS_PROMPT` — the per-chunk extraction prompt, shared with
  `ocr_app/scripts/ocr_server.py` so the notebook and the deployed server
  ask the model the exact same question.

Two small wrappers stay in the notebook (`run_vlm`, `parse_vlm_json`)
— they're how the demo talks directly to the vLLM endpoint.


In [ ]:
import sys
from pathlib import Path

# Locate repo root regardless of where the notebook is opened from
# (cloned working copy vs. /ocr/repo symlink to a tarball).
_candidates = [Path.cwd().parents[1], Path.cwd().parent, Path("/ocr/repo")]
REPO_ROOT = next((p for p in _candidates if (p / "ocr_app" / "scripts" / "merge.py").exists()), None)
assert REPO_ROOT is not None, f"Could not find ocr_app/scripts/. Tried: {_candidates}"
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print(f"Repo root: {REPO_ROOT}")

from ocr_app.scripts.chunk_extract import chunk_page_ranges, build_chunk_messages
from ocr_app.scripts.merge import merge_chunks
from ocr_app.scripts.doc_prompt import DOC_SYNTHESIS_PROMPT
print("Imports OK.")


### Notebook-side wrappers: `run_vlm` and `parse_vlm_json`

`run_vlm` is the OpenAI-compatible client this notebook uses to talk to
vLLM. It uses **streaming**: instead of the server holding the entire
response until generation is done and then sending it in one shot, it
sends each token (or small batch of tokens) over the wire as soon as
it's produced — and the client stitches the pieces back together.

We stream for one specific reason. The cluster's HTTP gateway has a
~3-minute idle timeout, and a single dense 20-page chunk can take 4+
minutes of generation before the first byte would otherwise come back.
Without streaming, the gateway sees no traffic and kills the connection
mid-generation. With streaming, tokens flow continuously so the
connection never goes idle.

`parse_vlm_json` strips markdown fences and tolerates the occasional
malformed tail. With vLLM's guided JSON decoding, parse failures are rare
in practice.


In [ ]:
import base64, io, json
from PIL import Image


# Public-ingress request-body cap (laptop_vpn path). The Knative gateway
# limits chat-completions POST bodies to ~1 MB; lossless PNGs of high-DPI
# page images blow past that even after a single downscale. On laptop_vpn
# we both shrink the image AND switch to JPEG (5–10x smaller for typical
# document pages with no perceptible quality loss for text). In-cluster
# calls bypass the ingress, so we keep PNG there for lossless fidelity.
_LAPTOP_VPN_MAX_IMAGE_DIM = 900   # px on the longest edge
_LAPTOP_VPN_JPEG_QUALITY  = 85    # visually indistinguishable from lossless for text


def _encode_image_b64(image: Image.Image) -> str:
    """Serialize a PIL image as a base64 image data URI for the VLM payload.

    The VLM endpoint takes images via OpenAI's `image_url` field. On
    in_cluster runs we send PNG as-is. On laptop_vpn runs we downscale +
    re-encode as JPEG so the request body fits under the public ingress's
    ~1 MB cap.
    """
    if RUN_LOCATION == "laptop_vpn":
        if max(image.size) > _LAPTOP_VPN_MAX_IMAGE_DIM:
            scale = _LAPTOP_VPN_MAX_IMAGE_DIM / max(image.size)
            image = image.resize(
                (int(image.width * scale), int(image.height * scale)),
                Image.LANCZOS,
            )
        buf = io.BytesIO()
        image.convert("RGB").save(
            buf, format="JPEG",
            quality=_LAPTOP_VPN_JPEG_QUALITY, optimize=True,
        )
        return f"data:image/jpeg;base64,{base64.b64encode(buf.getvalue()).decode()}"

    buf = io.BytesIO()
    image.save(buf, format="PNG")
    return f"data:image/png;base64,{base64.b64encode(buf.getvalue()).decode()}"


def run_vlm(messages: list, max_tokens: int = VLLM_MAX_TOKENS) -> str:
    """Send chat messages to vLLM and return the assembled response text.

    Role: thin OpenAI-compatible client tailored to this notebook's
    usage. Streams the response so the cluster gateway's ~3-minute
    idle timeout doesn't kill long generations mid-flight, and forces
    guided JSON output at the token level so we don't have to clean
    up smart quotes / trailing prose afterwards.
    """
    # Convert our internal {type:"image", image:...} to OpenAI image_url shape.
    oai_messages = []
    for msg in messages:
        content = []
        for part in msg["content"]:
            if part["type"] == "text":
                content.append({"type": "text", "text": part["text"]})
            elif part["type"] == "image":
                content.append({"type": "image_url", "image_url": {"url": part["image"]}})
        oai_messages.append({"role": msg["role"], "content": content})

    payload = {
        "model": MODEL_NAME,
        "messages": oai_messages,
        "max_tokens": max_tokens,
        "temperature": 0.0,
        "stream": True,
        "stream_options": {"include_usage": True},
        # Force valid JSON at the token level — eliminates whole classes
        # of cleanup work (curly quotes, JS-style comments, trailing prose).
        "response_format": {"type": "json_object"},
    }
    parts = []
    with httpx.stream(
        "POST",
        f"{VLLM_BASE_URL}/chat/completions",
        json=payload,
        timeout=httpx.Timeout(connect=30.0, read=720.0, write=60.0, pool=30.0),
    ) as resp:
        resp.raise_for_status()
        for line in resp.iter_lines():
            if not line.startswith("data:"):
                continue
            data = line[5:].strip()
            if data == "[DONE]":
                break
            event = json.loads(data)
            for choice in event.get("choices") or []:
                piece = (choice.get("delta") or {}).get("content")
                if piece:
                    parts.append(piece)
    return "".join(parts)


def parse_vlm_json(raw: str):
    """Turn a raw VLM response string into a Python dict.

    Role: defensive JSON parser for VLM output. Strips the optional
    ```json ... ``` markdown fence the model sometimes wraps the JSON
    in, tries strict parsing first, then falls back to extracting the
    outermost balanced `{...}` if that fails.

    Returns ``(parsed_dict, None)`` on success, ``(None, error_msg)``
    on failure — callers branch on the second value.
    """
    cleaned = raw.strip()
    if cleaned.startswith("```"):
        cleaned = cleaned.split("\n", 1)[1].rsplit("```", 1)[0].strip()
    try:
        return json.loads(cleaned), None
    except json.JSONDecodeError as e:
        # Last-ditch: trim to the outermost balanced object.
        first, last = cleaned.find("{"), cleaned.rfind("}")
        if first != -1 and last > first:
            try:
                return json.loads(cleaned[first:last + 1]), None
            except json.JSONDecodeError:
                pass
        return None, str(e)


print("Wrappers ready.")


## 3. Per-chunk extraction prompt

`DOC_SYNTHESIS_PROMPT` is the heart of the system: it tells the VLM what
schema to emit per chunk (narratives verbatim, three table classifications,
stakeholders, addresses, signatures, watermarks, etc.) and how to flag
content that continues across chunk boundaries.

Tweaking the prompt is the highest-leverage change you can make to the
output. Edit it in `ocr_app/scripts/doc_prompt.py`, restart the kernel,
and the change flows through both this notebook and the production
batch script.


In [ ]:
print(f"Prompt length: {len(DOC_SYNTHESIS_PROMPT)} chars")
print()
print(DOC_SYNTHESIS_PROMPT[:1500])
print("...")


## 4. Pick a sample document

Upload PDFs to `/ocr/` from the JupyterLab file browser. We grab the
first one for the live walkthrough; the batch step at the bottom processes
everything in the folder.


In [ ]:
# Paths pick themselves from RUN_LOCATION at the top of the notebook.
# `laptop_vpn` looks for PDFs in `data/rsp/` at the repo root — create
# that folder and drop your sample PDFs in if you don't have it yet. Outputs
# from the batch step land in `<ocr_dir>/vlm_output/`.
OCR_DIRS = {
    "in_cluster": Path("/ocr"),
    "laptop_vpn": Path("../../data/rsp"),
}
ocr_dir = OCR_DIRS[RUN_LOCATION]
OUT_DIR = ocr_dir / "output"
OUT_DIR.mkdir(exist_ok=True)

pdfs = sorted(ocr_dir.glob("*.pdf"))
assert pdfs, f"No PDFs in {ocr_dir}/. Upload one first."

for p in pdfs:
    print(f"  {p.name} ({p.stat().st_size / 1024:.0f} KB)")

DOC_PATH = pdfs[0]
print(f"\nUsing: {DOC_PATH.name}")
print(f"Output dir: {OUT_DIR}")


## 5. Filename metadata parser

Grant-admin docs follow a structured filename convention. Take this example:

```
A_RSP Public_AWD-001064_3718296725__AWD00000002__A_RSP_Award_Notice_of_Award.pdf
```

It splits into seven fields, joined by single `_`. A doubled `__` marks a
field that's intentionally empty:

| Field          | Value                          |
|----------------|--------------------------------|
| `Drawer`       | `A_RSP Public`                 |
| `AwardID`      | `AWD-001064`                   |
| `Field2`       | `3718296725`                   |
| `Field3`       | *(empty — the first `__`)*     |
| `Field4`       | `AWD00000002`                  |
| `Field5`       | *(empty — the second `__`)*    |
| `DocumentType` | `A_RSP_Award_Notice_of_Award`  |

We pull these into a `FileNameMetaData` block on the doc-level output.
Useful for downstream cataloging — and a good reminder that some metadata
is cheaper to extract from the filename than from the page.


In [ ]:
import re


def parse_filename(filename: str) -> dict:
    """Decode the structured grant-admin filename into FileNameMetaData.

    Role: cheap metadata source. Grant-admin PDFs follow a six-field
    naming convention (drawer, AwardID, four free-form fields,
    DocumentType) joined by single `_`, with `__` marking a field that
    is intentionally empty. Pulling these out of the filename is
    faster and more reliable than asking the VLM to re-read them off
    the page.

    Returns a flat dict ready to embed under `file_metadata` on the
    doc-level output.
    """
    stem = Path(filename).stem
    ext = Path(filename).suffix.lstrip(".")

    awd = re.search(r"_AWD-", stem)
    if not awd:
        return {"Drawer": stem, "AwardID": "", "Field2": "", "Field3": "",
                "Field4": "", "Field5": "", "DocumentType": stem,
                "DocumentTypeShort": stem, "FileType": ext}

    drawer = stem[:awd.start()]
    parts = stem[awd.start() + 1:].split("_")
    fields = parts[1:5] + [""] * max(0, 5 - len(parts[1:5]))
    doc_type = "_".join(parts[5:]) if len(parts) > 5 else ""
    short = doc_type
    for cat in ("_Award_", "_Budget_", "_Report_", "_Agreement_", "_Proposal_"):
        i = doc_type.find(cat)
        if i >= 0:
            short = doc_type[i + len(cat):]
            break

    return {"Drawer": drawer, "AwardID": parts[0],
            "Field2": fields[0], "Field3": fields[1],
            "Field4": fields[2], "Field5": fields[3],
            "DocumentType": doc_type, "DocumentTypeShort": short,
            "FileType": ext}


for k, v in parse_filename(DOC_PATH.name).items():
    print(f"  {k}: {v!r}")


## 6. Render the document as page images

Every page goes to the VLM as an image — that's how we capture layout,
tables, signatures, and watermarks that pure-text extraction would miss.
We render at 2x scale (~144 DPI), which is the sweet spot for dense
grant-admin tables: high enough that small numbers stay legible, low
enough that input tokens don't blow up.


In [ ]:
import fitz

doc = fitz.open(str(DOC_PATH))
page_images = []
for i, page in enumerate(doc):
    pix = page.get_pixmap(matrix=fitz.Matrix(2.0, 2.0))
    page_images.append(Image.frombytes("RGB", [pix.width, pix.height], pix.samples))
doc.close()

print(f"{DOC_PATH.name}: {len(page_images)} page(s)")
print(f"First page: {page_images[0].width}x{page_images[0].height}")
display(page_images[0].resize((page_images[0].width // 3, page_images[0].height // 3)))


## 7. Run the pipeline on this PDF

We've connected, picked a sample, and rendered its pages. Now drive
`DOC_PATH` through the full extraction pipeline, one stage per cell, so
each step's output is visible:

1. Plan the chunk ranges over the rendered pages.
2. Extract chunk 1 and inspect what came back (the fast-feedback
   sanity check).
3. Extract any remaining chunks (resume-aware: cached chunks from
   step 2 are reused).
4. Merge the chunk JSONs into one doc-level JSON.
5. Pass-2 synthesis to fill the doc-level summary.
6. Save the final JSON.

The batch cell at the bottom does these same six steps in a loop over
every PDF in `ocr_dir`.


### Why chunk at all?

Ideally we'd send the whole document in one VLM call — no chunk
boundaries, no merge step, no continuation-flag plumbing. Two reasons
we don't:

- **Operator-set context cap.** The vLLM server is started with
  `--max-model-len 16384`, which caps total prompt + output tokens. This
  is a deploy-time choice (set in the inference workload's vLLM args),
  *not* a property of the AWQ build — Qwen3-VL-32B's architectural
  ceiling is much higher. We pick a smaller cap because KV cache scales
  linearly with `max-model-len`, so larger values eat into the GPU
  memory available for batching and concurrent users. Twenty
  image-rendered pages with cross-page context already approach that
  16K ceiling for dense grant-admin material.
- **Long-context degradation is a known LLM failure mode.** Models
  routinely use their advertised context window worse than its raw
  size suggests — "lost in the middle" effects (Liu et al. 2023),
  accuracy falloff well before the hard token cap (RULER, NIAH-style
  benchmarks), and increased hallucination on long inputs are all
  documented in the literature. For this VLM on dense grant-admin
  pages, hands-on testing puts the practical knee around ~20 pages:
  silent omissions in `tables`, mid-table truncation, hallucinated
  continuations. Larger / more capable models likely push that
  boundary further out, but the underlying phenomenon is systemic,
  not a quirk of this stack.

So **20 pages with 1-page overlap** is the practical compromise: small
enough that per-chunk quality stays high, large enough that boundary
artifacts (split tables, narrative spillover) are manageable for the
merge step to clean up. If you swap in a stronger model, try bumping
`MAX_PAGES_PER_CHUNK` and watching for `finish_reason=="length"` in
the per-chunk debug output — that's the signal you've gone too big.


### Step 1 — plan the chunk ranges

`chunk_page_ranges(N, chunk_size, overlap)` returns half-open
`(start, end)` ranges covering all `N` rendered pages. Overlap=1 means
each adjacent pair of chunks shares one page so the merge step has a
deterministic stitch point.


In [ ]:
CHUNK_SIZES = {"in_cluster": 20, "laptop_vpn": 5}
MAX_PAGES_PER_CHUNK = CHUNK_SIZES[RUN_LOCATION]
CHUNK_OVERLAP = 1

ranges = chunk_page_ranges(len(page_images), MAX_PAGES_PER_CHUNK, CHUNK_OVERLAP)
print(f"{len(page_images)} page(s) → {len(ranges)} chunk(s):")
for ci, (s, e) in enumerate(ranges):
    print(f"  chunk {ci+1}: pages {s+1}-{e} ({e-s} pages)")


### Step 2 — extract chunk 1 and inspect it

Run the first planned chunk on its own first — it's a quick sanity
check that the prompt is producing the expected schema before we
spend time on the rest of the doc. The same `cache_file.exists()`
short-circuit will let Step 3 skip this chunk.


In [ ]:
import time

chunks_dir = OUT_DIR / f"{DOC_PATH.stem}_chunks"
chunks_dir.mkdir(exist_ok=True)

start, end = ranges[0]
cache_file = chunks_dir / "chunk_001.json"

if cache_file.exists():
    record = json.loads(cache_file.read_text(encoding="utf-8"))
    print(f"  chunk 1: cached")
else:
    msgs = build_chunk_messages(
        images=page_images[start:end],
        prompt=DOC_SYNTHESIS_PROMPT,
        encode_image_b64=_encode_image_b64,
        filename=DOC_PATH.name,
        is_first_chunk=True,
        is_last_chunk=(len(ranges) == 1),
        first_pdf_page=start + 1,
    )
    t0 = time.time()
    raw = run_vlm(msgs)
    elapsed = time.time() - t0
    extracted, err = parse_vlm_json(raw)
    assert extracted is not None, f"chunk 1 PARSE FAILED ({err})"
    record = {
        "chunk_index": 0, "page_start": start, "page_end": end,
        "experiment": {
            "model": MODEL_NAME, "elapsed_ms": round(elapsed * 1000, 1),
            "max_pages_per_chunk": MAX_PAGES_PER_CHUNK,
            "chunk_overlap": CHUNK_OVERLAP, "render_scale": 2.0,
        },
        "extracted": extracted,
    }
    cache_file.write_text(json.dumps(record, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"  chunk 1 (pages {start+1}-{end}): {elapsed:.1f}s")

ex = record["extracted"]
print(f"\nTop-level fields: {list(ex.keys())}")
print(f"  one_sentence_summary: {ex.get('one_sentence_summary', '')[:120]}")
print(f"  tables:              {len(ex.get('tables') or [])}")
print(f"  narrative_responses: {len(ex.get('narrative_responses') or [])}")
print(f"  stakeholders:        {len(ex.get('stakeholders') or [])}")
print(f"  addresses:           {len(ex.get('addresses') or [])}")
print(f"  document_tags:       {ex.get('document_tags') or []}")


### Step 3 — extract any remaining chunks

Loop over the full range list. The `cache_file.exists()` short-circuit
reuses chunk 1's cached result; everything from chunk 2 onward gets a
fresh VLM call. Chunk 1's `document_details` is forwarded to later
chunks as grounding (`forward_context`) so the model keeps a stable
view of the document.


In [ ]:
chunk_records = []
forward_ctx = None
for ci, (start, end) in enumerate(ranges):
    cache_file = chunks_dir / f"chunk_{ci+1:03d}.json"
    if cache_file.exists():
        cached = json.loads(cache_file.read_text(encoding="utf-8"))
        chunk_records.append(cached)
        if ci == 0:
            ex = cached.get("extracted") or {}
            forward_ctx = {
                "document_details": ex.get("document_details") or {},
                "document_tags": ex.get("document_tags") or [],
                "one_sentence_summary": ex.get("one_sentence_summary") or "",
            }
        print(f"  chunk {ci+1}: cached")
        continue

    msgs = build_chunk_messages(
        images=page_images[start:end],
        prompt=DOC_SYNTHESIS_PROMPT,
        encode_image_b64=_encode_image_b64,
        filename=DOC_PATH.name,
        is_first_chunk=(ci == 0),
        is_last_chunk=(ci == len(ranges) - 1),
        pinned_images=[(page_images[0], "Page 1 (cover)")] if ci > 0 else None,
        forward_context=forward_ctx if ci > 0 else None,
        first_pdf_page=start + 1,
    )
    t0 = time.time()
    raw = run_vlm(msgs)
    elapsed = time.time() - t0
    extracted, err = parse_vlm_json(raw)
    if extracted is None:
        print(f"  chunk {ci+1}: PARSE FAILED ({err}) — skipping")
        continue
    record = {
        "chunk_index": ci, "page_start": start, "page_end": end,
        "experiment": {
            "model": MODEL_NAME, "elapsed_ms": round(elapsed * 1000, 1),
            "max_pages_per_chunk": MAX_PAGES_PER_CHUNK,
            "chunk_overlap": CHUNK_OVERLAP, "render_scale": 2.0,
        },
        "extracted": extracted,
    }
    cache_file.write_text(json.dumps(record, indent=2, ensure_ascii=False), encoding="utf-8")
    chunk_records.append(record)
    if ci == 0:
        forward_ctx = {
            "document_details": extracted.get("document_details") or {},
            "document_tags": extracted.get("document_tags") or [],
            "one_sentence_summary": extracted.get("one_sentence_summary") or "",
        }
    print(f"  chunk {ci+1}/{len(ranges)} (pages {start+1}-{end}): {elapsed:.1f}s")

print(f"\n{len(chunk_records)} chunk(s) extracted, cached under {chunks_dir.name}/")


### Step 4 — merge chunks into one doc-level JSON


### How the merge works

Each chunk is an independent VLM call — it doesn't see what other chunks
extracted. `merge_chunks` reconciles those independent outputs into one
doc-level JSON using a few rules:

- **Dedup by stable identity.** Each stakeholder, address, table, and
  narrative gets a content fingerprint (stakeholders by
  `full_name + institution + role`; addresses by
  `address_line1 + city + postal_code`; tables by header signature plus
  first-row text; narratives by preceding section header plus a normalized
  prefix of the verbatim text). Items with the same fingerprint collapse
  into one — that's how the 1-page chunk overlap doesn't produce 2x
  duplicates.
- **Stitch continuation spans.** A table flagged
  `continues_to_next_chunk` in chunk N is joined with the table flagged
  `continues_from_previous_chunk` in chunk N+1, provided their preceding
  section headers match. Same rule for verbatim narrative responses that
  span pages.
- **Coalesce fields when items merge.** When two records dedupe to one,
  the merge picks non-empty values field-by-field — it never overwrites
  a populated field with an empty one. For stakeholders specifically,
  the merge prefers the *more complete* record (more populated fields
  wins) when both sides have data.
- **Roll up document-level fields.** `document_details` (title, ID,
  amount, etc.) takes the first non-empty value seen across chunks.
  `document_tags` is the set union.
- **Lint the result.** The merge also flags telltale problems —
  continuation flags still set after stitching, near-duplicate tables
  that didn't dedupe, empty narrative bodies — into `potential_issues`.
  Pass 2 appends any VLM-spotted issues to that same list.

That's why chunk overlap and the per-chunk `continues_to_*` flags exist:
the merge step needs deterministic signals to know when two items in
different chunks are actually the same content.


In [ ]:
merged = merge_chunks(chunk_records, extraction_prompt=DOC_SYNTHESIS_PROMPT)

fmeta = parse_filename(DOC_PATH.name)
fmeta["page_count"] = len(page_images)
merged = {"filename": DOC_PATH.name, "file_metadata": fmeta, **merged}

print(f"Merged doc-level JSON:")
print(f"  tables:              {len(merged.get('tables') or [])}")
print(f"  narrative_responses: {len(merged.get('narrative_responses') or [])}")
print(f"  stakeholders:        {len(merged.get('stakeholders') or [])}")
print(f"  addresses:           {len(merged.get('addresses') or [])}")
print(f"  potential_issues:    {len(merged.get('potential_issues') or [])}")


### Step 5 — pass-2 synthesis

Pass 2 is the second (and final) VLM call per document. It's text-only:
we feed the model a compacted view of the merged JSON (per-chunk
summaries + item counts, no images) and it returns a doc-level
`one_sentence_summary`, a `confidence_narrative`, and a list of
`potential_issues` the deterministic merge couldn't see.

We define its prompt and a small `run_pass2(merged)` wrapper here so
the call below — and the batch loop at the bottom — both use the
exact same code path.


In [ ]:
SYNTHESIS_PROMPT = """You are given a MERGED doc-level JSON from a
multi-page grant-administration document. Pass 1 already extracted
tables, narratives, stakeholders, and addresses chunk-by-chunk and a
deterministic merge has stitched cross-chunk spans and deduped items.
Your ONLY job is to fill two summary fields and optionally flag issues.

The merged JSON includes a ``chunks`` array. Each entry has a brief
per-chunk ``extracted.one_sentence_summary`` and
``extracted.confidence_narrative`` — USE those as the raw material for
your doc-level summaries. Do not re-read the full tables/narratives.

Return a JSON object with EXACTLY these fields:
{
  "one_sentence_summary": "<one-sentence doc-level summary>",
  "confidence_narrative": "<3–5 sentences summarizing extraction quality>",
  "potential_issues": ["<optional flags for merge oddities>"]
}

Output ONLY valid JSON. No markdown fences."""


In [ ]:
def run_pass2(merged: dict) -> dict | None:
    """Run pass 2: ask the VLM for a doc-level summary of the merged extraction.

    Role: the second (and final) VLM call per document. Inputs are
    text-only — a compacted view of the merged JSON (per-chunk
    summaries, confidence narratives, item counts), no images.
    Returns the parsed `{one_sentence_summary, confidence_narrative,
    potential_issues}` block, or None when the call or JSON parse
    fails (callers leave pass 1's output untouched in that case).
    """
    chunks = [
        {
            "page_range": [c.get("page_start"), c.get("page_end")],
            "one_sentence_summary": (c.get("extracted") or {}).get("one_sentence_summary", ""),
            "confidence_narrative": (c.get("extracted") or {}).get("confidence_narrative", ""),
        }
        for c in (merged.get("chunks") or [])
    ]
    compact = {
        "document_details": merged.get("document_details") or {},
        "document_tags": merged.get("document_tags") or [],
        "item_counts": {
            "tables": len(merged.get("tables") or []),
            "narrative_responses": len(merged.get("narrative_responses") or []),
            "stakeholders": len(merged.get("stakeholders") or []),
            "addresses": len(merged.get("addresses") or []),
        },
        "potential_issues": list(merged.get("potential_issues") or []),
        "chunks": chunks,
    }
    body = json.dumps(compact, indent=1, ensure_ascii=False)
    msgs = [{"role": "user", "content": [
        {"type": "text", "text": f"{SYNTHESIS_PROMPT}\n\n{body}"}
    ]}]
    raw = run_vlm(msgs, max_tokens=6000)
    parsed, _ = parse_vlm_json(raw)
    return parsed


In [ ]:
synthesis = run_pass2(merged)

if synthesis:
    if synthesis.get("one_sentence_summary"):
        merged["one_sentence_summary"] = synthesis["one_sentence_summary"]
    if synthesis.get("confidence_narrative"):
        merged["confidence_narrative"] = synthesis["confidence_narrative"]
    existing = list(merged.get("potential_issues") or [])
    for note in synthesis.get("potential_issues") or []:
        if note and note not in existing:
            existing.append(note)
    merged["potential_issues"] = existing
    print(f"Pass 2 done.")
    print(f"  summary: {merged['one_sentence_summary'][:120]!r}")
    print(f"  potential_issues: {len(merged.get('potential_issues') or [])}")
else:
    print("Pass 2 failed — keeping pass 1 only.")


### Step 6 — save the final JSON

Drop the `chunks[]` sidecar (it was only kept for pass-2 input) and
write the result to `<OUT_DIR>/<doc-stem>_extracted.json`.


In [ ]:
out_path = OUT_DIR / f"{DOC_PATH.stem}_extracted.json"
final = {k: v for k, v in merged.items() if k != "chunks"}
out_path.write_text(json.dumps(final, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(f"Saved: {out_path}")


## 8. Batch convenience: every PDF in the folder

Same five steps wrapped in a `for pdf in pdfs:` loop, with `SKIP_EXISTING`
so reruns don't reprocess docs that already have output. The
`run_pass2` helper and `SYNTHESIS_PROMPT` from above are reused as-is.


In [ ]:
# ── Tunables ────────────────────────────────────────────────────────
# ocr_dir + OUT_DIR were set above based on RUN_LOCATION. Chunk size is also
# RUN_LOCATION-keyed: laptop_vpn requests have to fit under the Knative
# ingress's ~1 MB body cap, so we send fewer image-pages per call there.
CHUNK_SIZES = {"in_cluster": 20, "laptop_vpn": 5}
MAX_PAGES_PER_CHUNK = CHUNK_SIZES[RUN_LOCATION]
CHUNK_OVERLAP = 1
SKIP_EXISTING = True

pdf_files = sorted(ocr_dir.glob("*.pdf"), key=lambda p: p.stat().st_size, reverse=True)
print(f"{len(pdf_files)} PDF(s) to process")

for pdf_path in pdf_files:
    out_path = OUT_DIR / f"{pdf_path.stem}_extracted.json"
    chunks_dir = OUT_DIR / f"{pdf_path.stem}_chunks"

    if SKIP_EXISTING and out_path.exists():
        print(f"SKIP (already done): {pdf_path.name}")
        continue

    print(f"\n=== {pdf_path.name} ===")
    chunks_dir.mkdir(exist_ok=True)

    doc = fitz.open(str(pdf_path))
    images = []
    for page in doc:
        pix = page.get_pixmap(matrix=fitz.Matrix(2.0, 2.0))
        images.append(Image.frombytes("RGB", [pix.width, pix.height], pix.samples))
    doc.close()

    ranges = chunk_page_ranges(len(images), MAX_PAGES_PER_CHUNK, CHUNK_OVERLAP)
    print(f"  {len(images)} pages → {len(ranges)} chunk(s)")

    chunk_records = []
    forward_ctx = None
    for ci, (start, end) in enumerate(ranges):
        cache_file = chunks_dir / f"chunk_{ci+1:03d}.json"
        if cache_file.exists():
            chunk_records.append(json.loads(cache_file.read_text(encoding="utf-8")))
            print(f"  chunk {ci+1}: cached")
            continue

        msgs = build_chunk_messages(
            images=images[start:end],
            prompt=DOC_SYNTHESIS_PROMPT,
            encode_image_b64=_encode_image_b64,
            filename=pdf_path.name,
            is_first_chunk=(ci == 0),
            is_last_chunk=(ci == len(ranges) - 1),
            pinned_images=[(images[0], "Page 1 (cover)")] if ci > 0 else None,
            forward_context=forward_ctx if ci > 0 else None,
            first_pdf_page=start + 1,
        )
        t0 = time.time()
        raw = run_vlm(msgs)
        elapsed = time.time() - t0
        extracted, err = parse_vlm_json(raw)
        if extracted is None:
            print(f"  chunk {ci+1}: PARSE FAILED ({err}) — skipping")
            continue

        record = {
            "chunk_index": ci, "page_start": start, "page_end": end,
            "experiment": {
                "model": MODEL_NAME, "elapsed_ms": round(elapsed * 1000, 1),
                "max_pages_per_chunk": MAX_PAGES_PER_CHUNK,
                "chunk_overlap": CHUNK_OVERLAP, "render_scale": 2.0,
            },
            "extracted": extracted,
        }
        cache_file.write_text(json.dumps(record, indent=2, ensure_ascii=False), encoding="utf-8")
        chunk_records.append(record)
        if ci == 0:
            forward_ctx = {
                "document_details": extracted.get("document_details") or {},
                "document_tags": extracted.get("document_tags") or [],
                "one_sentence_summary": extracted.get("one_sentence_summary") or "",
            }
        print(f"  chunk {ci+1}/{len(ranges)} (pages {start+1}-{end}): {elapsed:.1f}s")

    if not chunk_records:
        print(f"  no chunks succeeded — skipping merge")
        continue

    merged = merge_chunks(chunk_records, extraction_prompt=DOC_SYNTHESIS_PROMPT)
    fmeta = parse_filename(pdf_path.name)
    fmeta["page_count"] = len(images)
    merged = {"filename": pdf_path.name, "file_metadata": fmeta, **merged}
    print(f"  merged: {len(merged.get('tables') or [])} tables, "
          f"{len(merged.get('narrative_responses') or [])} narratives, "
          f"{len(merged.get('stakeholders') or [])} stakeholders")

    synthesis = run_pass2(merged)
    if synthesis:
        if synthesis.get("one_sentence_summary"):
            merged["one_sentence_summary"] = synthesis["one_sentence_summary"]
        if synthesis.get("confidence_narrative"):
            merged["confidence_narrative"] = synthesis["confidence_narrative"]
        existing = list(merged.get("potential_issues") or [])
        for note in synthesis.get("potential_issues") or []:
            if note and note not in existing:
                existing.append(note)
        merged["potential_issues"] = existing
        print(f"  pass 2: {merged['one_sentence_summary'][:90]!r}")

    final = {k: v for k, v in merged.items() if k != "chunks"}
    out_path.write_text(json.dumps(final, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
    print(f"  → {out_path.name}")

print("\nBatch complete.")


## 9. Inspect a result

Look at the doc-level JSON produced by the batch. Useful fields to sanity
check:

- `one_sentence_summary` — the pass-2 doc summary
- `file_metadata` — parsed filename
- `tables` / `narrative_responses` / `stakeholders` / `addresses` — the
  extracted content
- `potential_issues` — merge lint + pass-2 flags


In [ ]:
out_files = sorted(OUT_DIR.glob("*_extracted.json"))
assert out_files, f"No outputs in {OUT_DIR}/. Run the batch cell first."
sample = json.loads(out_files[0].read_text(encoding="utf-8"))

print(f"=== {sample['filename']} ===")
print(f"summary: {sample.get('one_sentence_summary', '')}")
print(f"\ndocument_details: {json.dumps(sample.get('document_details', {}), indent=2)}")
print(f"\ndocument_tags: {sample.get('document_tags', [])}")
print(f"\ncounts:")
print(f"  tables:              {len(sample.get('tables') or [])}")
print(f"  narrative_responses: {len(sample.get('narrative_responses') or [])}")
print(f"  stakeholders:        {len(sample.get('stakeholders') or [])}")
print(f"  addresses:           {len(sample.get('addresses') or [])}")
issues = sample.get("potential_issues") or []
print(f"\npotential_issues: {len(issues)}")
for issue in issues[:5]:
    print(f"  • {issue}")
